In [1]:
import pandas as pd
import json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set style for better-looking plots
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")


In [2]:
# Load all 4 datasets
def load_jsonl(filename):
    data = []
    with open(filename, 'r') as f:
        for line in f:
            data.append(json.loads(line))
    return pd.DataFrame(data)

# Load datasets
df_jml_gpt4 = load_jsonl('jml_gpt4.jsonl')
df_jml_sonnet = load_jsonl('jml_sonnet.jsonl')
df_maple_gpt4 = load_jsonl('maple_gpt4.jsonl')
df_maple_sonnet = load_jsonl('maple_sonnet.jsonl')

print(f"JML GPT-4 shape: {df_jml_gpt4.shape}")
print(f"JML Sonnet shape: {df_jml_sonnet.shape}")
print(f"Maple GPT-4 shape: {df_maple_gpt4.shape}")
print(f"Maple Sonnet shape: {df_maple_sonnet.shape}")

# Combine class_name and feature_name into a single identifier column
for df in [df_jml_gpt4, df_jml_sonnet, df_maple_gpt4, df_maple_sonnet]:
    df['class_feature'] = df['class_name'] + '.' + df['feature_name']
    # Drop the original separate columns
    df.drop(columns=['class_name', 'feature_name'], inplace=True)

# Add dataset identifier to each dataframe
df_jml_gpt4['dataset'] = 'JML'
df_jml_sonnet['dataset'] = 'JML'
df_maple_gpt4['dataset'] = 'Maple'
df_maple_sonnet['dataset'] = 'Maple'

# Extract AI request times and verification times from interactions
def extract_ai_times(df):
    """Extract total AI request time and verification time from interactions array"""
    ai_request_times = []
    verification_times = []
    
    for idx, row in df.iterrows():
        interactions = row.get('interactions', [])
        if isinstance(interactions, list):
            total_ai_time = 0.0
            total_verification_time = 0.0
            for interaction in interactions:
                if isinstance(interaction, dict):
                    total_ai_time += interaction.get('ai_request_time_seconds', 0.0) or 0.0
                    total_verification_time += interaction.get('verification_time_seconds', 0.0) or 0.0
            ai_request_times.append(total_ai_time)
            verification_times.append(total_verification_time)
        else:
            ai_request_times.append(0.0)
            verification_times.append(0.0)
    
    df['total_ai_request_time_seconds'] = ai_request_times
    df['total_verification_time_seconds'] = verification_times
    return df

# Extract times for all datasets
df_jml_gpt4 = extract_ai_times(df_jml_gpt4)
df_jml_sonnet = extract_ai_times(df_jml_sonnet)
df_maple_gpt4 = extract_ai_times(df_maple_gpt4)
df_maple_sonnet = extract_ai_times(df_maple_sonnet)

df_jml_gpt4.head()


JML GPT-4 shape: (10, 9)
JML Sonnet shape: (10, 9)
Maple GPT-4 shape: (26, 9)
Maple Sonnet shape: (26, 9)


,model,llm_interactions,success,max_retries_reached,final_status,interactions,total_elapsed_time_seconds,class_feature,dataset,total_ai_request_time_seconds,total_verification_time_seconds
0,gpt-4o-mini,4,True,False,Verification passed after 4 LLM interaction(s),"[{'interaction_number': 1, 'error_message': ' ...",42.355738,LCM_1.div,JML,22.626532,17.136095
1,gpt-4o-mini,1,True,False,Verification passed after 1 LLM interaction(s),"[{'interaction_number': 1, 'error_message': ' ...",26.921428,LCM_10.lcm,JML,21.554007,2.888423
2,gpt-4o-mini,1,True,False,Verification passed after 1 LLM interaction(s),"[{'interaction_number': 1, 'error_message': ' ...",10.506324,LCM_11.div,JML,5.066841,2.463777
3,gpt-4o-mini,1,True,False,Verification passed after 1 LLM interaction(s),"[{'interaction_number': 1, 'error_message': ' ...",27.180437,LCM_12.lcm,JML,21.402088,2.902696
4,gpt-4o-mini,1,True,False,Verification passed after 1 LLM interaction(s),"[{'interaction_number': 1, 'error_message': ' ...",27.973689,LCM_13.lcm,JML,22.429917,2.580856


In [3]:
df_maple_gpt4[df_maple_gpt4["success"]==False]

,model,llm_interactions,success,max_retries_reached,final_status,interactions,total_elapsed_time_seconds,class_feature,dataset,total_ai_request_time_seconds,total_verification_time_seconds
2,gpt-4o-mini,10,False,True,Max retries (10) reached,"[{'interaction_number': 1, 'error_message': ' ...",58.625858,MAPLE_RECURSIVE_CONSEQ_1.sum,Maple,54.212897,4.337310
3,gpt-4o-mini,10,False,True,Max retries (10) reached,"[{'interaction_number': 1, 'error_message': ' ...",59.102600,MAPLE_RECURSIVE_CONSEQ_2.sum,Maple,55.867339,3.137718
4,gpt-4o-mini,10,False,True,Max retries (10) reached,"[{'interaction_number': 1, 'error_message': ' ...",57.914921,MAPLE_RECURSIVE_CONSEQ_3.sum,Maple,54.607801,3.185183
5,gpt-4o-mini,10,False,True,Max retries (10) reached,"[{'interaction_number': 1, 'error_message': ' ...",66.723216,MAPLE_RECURSIVE_CONSEQ_4.sum,Maple,63.549553,3.089913
6,gpt-4o-mini,10,False,True,Max retries (10) reached,"[{'interaction_number': 1, 'error_message': ' ...",53.730110,MAPLE_RECURSIVE_INCREMENT_1.increment,Maple,50.502357,3.162956
7,gpt-4o-mini,10,False,True,Max retries (10) reached,"[{'interaction_number': 1, 'error_message': ' ...",58.523360,MAPLE_RECURSIVE_INCREMENT_2.increment,Maple,55.246705,3.204993
8,gpt-4o-mini,10,False,True,Max retries (10) reached,"[{'interaction_number': 1, 'error_message': ' ...",50.768066,MAPLE_RECURSIVE_INCREMENT_3.increment,Maple,47.518658,3.193759
9,gpt-4o-mini,10,False,True,Max retries (10) reached,"[{'interaction_number': 1, 'error_message': ' ...",50.934448,MAPLE_RECURSIVE_INCREMENT_4.increment,Maple,47.550635,3.290941
10,gpt-4o-mini,10,False,True,Max retries (10) reached,"[{'interaction_number': 1, 'error_message': ' ...",40.644691,MAPLE_RECURSIVE_MAX_2_1.max,Maple,37.218678,3.345491
11,gpt-4o-mini,10,False,True,Max retries (10) reached,"[{'interaction_number': 1, 'error_message': ' ...",40.621945,MAPLE_RECURSIVE_MAX_2_2.max,Maple,37.129137,3.408426


In [4]:
# Join within each dataset, then union the results
# JML dataset: Join GPT-4 and Sonnet
df_jml_gpt4_join = df_jml_gpt4[['class_feature', 'llm_interactions', 'total_elapsed_time_seconds', 
                                 'total_ai_request_time_seconds', 'total_verification_time_seconds', 'success']].copy()
df_jml_gpt4_join = df_jml_gpt4_join.rename(columns={
    'llm_interactions': 'llm_interactions_gpt4',
    'total_elapsed_time_seconds': 'time_seconds_gpt4',
    'total_ai_request_time_seconds': 'ai_request_time_gpt4',
    'total_verification_time_seconds': 'verification_time_gpt4',
    'success': 'success_gpt4'
})

df_jml_sonnet_join = df_jml_sonnet[['class_feature', 'llm_interactions', 'total_elapsed_time_seconds',
                                     'total_ai_request_time_seconds', 'total_verification_time_seconds', 'success']].copy()
df_jml_sonnet_join = df_jml_sonnet_join.rename(columns={
    'llm_interactions': 'llm_interactions_sonnet',
    'total_elapsed_time_seconds': 'time_seconds_sonnet',
    'total_ai_request_time_seconds': 'ai_request_time_sonnet',
    'total_verification_time_seconds': 'verification_time_sonnet',
    'success': 'success_sonnet'
})

df_jml_joined = pd.merge(df_jml_gpt4_join, df_jml_sonnet_join, on='class_feature', how='outer')
df_jml_joined['dataset'] = 'JML'
df_jml_joined['interaction_diff'] = df_jml_joined['llm_interactions_gpt4'] - df_jml_joined['llm_interactions_sonnet']
df_jml_joined['time_diff'] = df_jml_joined['time_seconds_gpt4'] - df_jml_joined['time_seconds_sonnet']
df_jml_joined['ai_request_time_diff'] = df_jml_joined['ai_request_time_gpt4'] - df_jml_joined['ai_request_time_sonnet']
df_jml_joined['verification_time_diff'] = df_jml_joined['verification_time_gpt4'] - df_jml_joined['verification_time_sonnet']

# Maple dataset: Join GPT-4 and Sonnet
df_maple_gpt4_join = df_maple_gpt4[['class_feature', 'llm_interactions', 'total_elapsed_time_seconds',
                                     'total_ai_request_time_seconds', 'total_verification_time_seconds', 'success']].copy()
df_maple_gpt4_join = df_maple_gpt4_join.rename(columns={
    'llm_interactions': 'llm_interactions_gpt4',
    'total_elapsed_time_seconds': 'time_seconds_gpt4',
    'total_ai_request_time_seconds': 'ai_request_time_gpt4',
    'total_verification_time_seconds': 'verification_time_gpt4',
    'success': 'success_gpt4'
})

df_maple_sonnet_join = df_maple_sonnet[['class_feature', 'llm_interactions', 'total_elapsed_time_seconds',
                                         'total_ai_request_time_seconds', 'total_verification_time_seconds', 'success']].copy()
df_maple_sonnet_join = df_maple_sonnet_join.rename(columns={
    'llm_interactions': 'llm_interactions_sonnet',
    'total_elapsed_time_seconds': 'time_seconds_sonnet',
    'total_ai_request_time_seconds': 'ai_request_time_sonnet',
    'total_verification_time_seconds': 'verification_time_sonnet',
    'success': 'success_sonnet'
})

df_maple_joined = pd.merge(df_maple_gpt4_join, df_maple_sonnet_join, on='class_feature', how='outer')
df_maple_joined['dataset'] = 'Maple'
df_maple_joined['interaction_diff'] = df_maple_joined['llm_interactions_gpt4'] - df_maple_joined['llm_interactions_sonnet']
df_maple_joined['time_diff'] = df_maple_joined['time_seconds_gpt4'] - df_maple_joined['time_seconds_sonnet']
df_maple_joined['ai_request_time_diff'] = df_maple_joined['ai_request_time_gpt4'] - df_maple_joined['ai_request_time_sonnet']
df_maple_joined['verification_time_diff'] = df_maple_joined['verification_time_gpt4'] - df_maple_joined['verification_time_sonnet']

# Union the two joined datasets
df_all = pd.concat([df_jml_joined, df_maple_joined], ignore_index=True)
df_all['interaction_diff'] = df_all['llm_interactions_gpt4'] - df_all['llm_interactions_sonnet']
df_all['time_diff'] = df_all['time_seconds_gpt4'] - df_all['time_seconds_sonnet']
df_all['ai_request_time_diff'] = df_all['ai_request_time_gpt4'] - df_all['ai_request_time_sonnet']
df_all['verification_time_diff'] = df_all['verification_time_gpt4'] - df_all['verification_time_sonnet']

print(f"JML joined shape: {df_jml_joined.shape}")
print(f"Maple joined shape: {df_maple_joined.shape}")
print(f"Combined (union) DataFrame shape: {df_all.shape}")
df_all.head()


JML joined shape: (10, 16)
Maple joined shape: (26, 16)
Combined (union) DataFrame shape: (36, 16)


,class_feature,llm_interactions_gpt4,time_seconds_gpt4,ai_request_time_gpt4,verification_time_gpt4,success_gpt4,llm_interactions_sonnet,time_seconds_sonnet,ai_request_time_sonnet,verification_time_sonnet,success_sonnet,dataset,interaction_diff,time_diff,ai_request_time_diff,verification_time_diff
0,LCM_1.div,4,42.355738,22.626532,17.136095,True,1,20.005055,7.170453,10.320707,True,JML,3,22.350683,15.456080,6.815388
1,LCM_10.lcm,1,26.921428,21.554007,2.888423,True,1,22.816788,17.461203,2.868860,True,JML,0,4.104639,4.092804,0.019563
2,LCM_11.div,1,10.506324,5.066841,2.463777,True,1,12.043649,7.071722,2.184680,True,JML,0,-1.537325,-2.004881,0.279096
3,LCM_12.lcm,1,27.180437,21.402088,2.902696,True,1,16.868136,11.590349,2.384943,True,JML,0,10.312301,9.811739,0.517754
4,LCM_13.lcm,1,27.973689,22.429917,2.580856,True,1,20.624958,15.336958,2.793884,True,JML,0,7.348731,7.092959,-0.213029


In [5]:
# Success Analysis
print("=" * 60)
print("SUCCESS ANALYSIS")
print("=" * 60)

# Overall success statistics
df_success_both = df_all.dropna(subset=['success_gpt4', 'success_sonnet'])

if len(df_success_both) > 0:
    gpt4_success = df_success_both['success_gpt4'].sum()
    sonnet_success = df_success_both['success_sonnet'].sum()
    total = len(df_success_both)
    
    print(f"\nOverall Success Rates (across both datasets):")
    print(f"Total cases with success data: {total}")
    print(f"GPT-4 success rate: {gpt4_success}/{total} ({gpt4_success/total*100:.1f}%)")
    print(f"Sonnet success rate: {sonnet_success}/{total} ({sonnet_success/total*100:.1f}%)")
    
    print(f"\nSuccess Breakdown:")
    both_succeeded = ((df_success_both['success_gpt4'] == True) & (df_success_both['success_sonnet'] == True)).sum()
    only_gpt4 = ((df_success_both['success_gpt4'] == True) & (df_success_both['success_sonnet'] == False)).sum()
    only_sonnet = ((df_success_both['success_gpt4'] == False) & (df_success_both['success_sonnet'] == True)).sum()
    both_failed = ((df_success_both['success_gpt4'] == False) & (df_success_both['success_sonnet'] == False)).sum()
    
    print(f"  Both succeeded: {both_succeeded} ({both_succeeded/total*100:.1f}%)")
    print(f"  Only GPT-4 succeeded: {only_gpt4} ({only_gpt4/total*100:.1f}%)")
    print(f"  Only Sonnet succeeded: {only_sonnet} ({only_sonnet/total*100:.1f}%)")
    print(f"  Both failed: {both_failed} ({both_failed/total*100:.1f}%)")
    
    # By dataset
    print(f"\n" + "-" * 60)
    print("Success Rates by Dataset:")
    print("-" * 60)
    
    for dataset_name, df_dataset in [('JML', df_jml_joined), ('Maple', df_maple_joined)]:
        df_ds_success = df_dataset.dropna(subset=['success_gpt4', 'success_sonnet'])
        if len(df_ds_success) > 0:
            gpt4_ds = df_ds_success['success_gpt4'].sum()
            sonnet_ds = df_ds_success['success_sonnet'].sum()
            total_ds = len(df_ds_success)
            print(f"\n{dataset_name} Dataset:")
            print(f"  GPT-4 success rate: {gpt4_ds}/{total_ds} ({gpt4_ds/total_ds*100:.1f}%)")
            print(f"  Sonnet success rate: {sonnet_ds}/{total_ds} ({sonnet_ds/total_ds*100:.1f}%)")
else:
    print("No success data available")


SUCCESS ANALYSIS

Overall Success Rates (across both datasets):
Total cases with success data: 36
GPT-4 success rate: 10/36 (27.8%)
Sonnet success rate: 36/36 (100.0%)

Success Breakdown:
  Both succeeded: 10 (27.8%)
  Only GPT-4 succeeded: 0 (0.0%)
  Only Sonnet succeeded: 26 (72.2%)
  Both failed: 0 (0.0%)

------------------------------------------------------------
Success Rates by Dataset:
------------------------------------------------------------

JML Dataset:
  GPT-4 success rate: 8/10 (80.0%)
  Sonnet success rate: 10/10 (100.0%)

Maple Dataset:
  GPT-4 success rate: 2/26 (7.7%)
  Sonnet success rate: 26/26 (100.0%)
